In [1]:
import pandas as pd

In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor, IsolationForest

import matplotlib.pyplot as plt

# ===============================
# 1. LOAD DATA
# ===============================
df = pd.read_csv("SphereSleep.sensor_data.csv")


In [3]:
import pandas as pd

threshold = 5  # set your threshold

df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp")

df = df[df['timestamp'].dt.year > 1970]

# --- 1. Define sleep session date ---
# Rule: 
# 19:00–23:59 → belongs to same day session 
# 00:00–06:00 → belongs to previous day's session

df["session_date"] = df["timestamp"].dt.date

df.loc[df["timestamp"].dt.hour < 6, "session_date"] = (
    df.loc[df["timestamp"].dt.hour < 6, "timestamp"] - pd.Timedelta(days=1)
).dt.date

# --- 2. Create sleep_session column ---
df["sleep_session"] = (
    df["user_id"].astype(str) + "_" + df["session_date"].astype(str)
)

In [4]:
# --- 3. Movement within each sleep session --- 
# In distance col, if prev row's value is different to current value, it adds "Y"
df["distance_diff"] = df.groupby("sleep_session")["distance"].diff().abs()

df["movement"] = df["distance_diff"].gt(threshold).map({True: "Y", False: "N"})

# first record per session = blank
df.loc[df.groupby("sleep_session").head(1).index, "movement"] = ""


In [6]:
df.columns

Index(['timestamp', 'user_id', '_id', 'light_intensity', 'temperature',
       'humidity', 'session_id', 'sound_level', 'distance', 'motion_detected',
       'session_date', 'sleep_session', 'distance_diff', 'movement'],
      dtype='object')

In [7]:
df.columns
df = df[['timestamp', 'user_id', '_id', 'light_intensity', 'temperature',
       'humidity',  'sound_level', 'distance', 'session_date',  'movement']]
df
df.to_csv("output_with_movement_within_sleep_session.csv", index=False)